## 长期记忆
### 基础api
- put
- get
- search

In [2]:
from typing import NotRequired

from dotenv import load_dotenv
from langgraph.store.memory import InMemoryStore
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

InMemoryStore 使用内存存储长期记忆,适合测试

In [3]:

IMStore = InMemoryStore()

namescape = ("users",)#元组
key = "user1"#字符串
value = {#json字典
    "name": "xiaoming",
    "age": 18,
    "gender": "male"
}
IMStore.put(namescape,key,value)

item = IMStore.get(namescape,key)
print(item) if item is not None else print("key not found")


Item(namespace=['users'], key='user1', value={'name': 'xiaoming', 'age': 18, 'gender': 'male'}, created_at='2026-08-25T05:38:42.531581+00:00', updated_at='2026-08-25T05:38:42.531583+00:00')


PostgresStore 用于在postgresql数据库中存储数据,适合生产环境

In [7]:
import os
from langgraph.store.postgres import PostgresStore

DB_URL = os.getenv("POSTGRES_DB_URL")

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    namescape = ("users",)#元组
    key = "user1"#字符串
    value = {#json字典
        "name": "xiaoming",
        "age": 18,
        "gender": "male"
    }
    store.put(namescape,key,value)

    item = store.get(namescape,key)
    print(item) if item is not None else print("key not found")


Item(namespace=['users'], key='user1', value={'age': 18, 'name': 'xiaoming', 'gender': 'male'}, created_at='2026-08-25T13:50:11.989998+08:00', updated_at='2026-08-25T13:50:11.989998+08:00')


### 在agent中使用长期记忆


In [10]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langchain.agents import create_agent, AgentState
from langgraph.store.memory import InMemoryStore
from typing import NotRequired

class SavaUserInfoInput(BaseModel):
    name: str = Field(description="用户姓名")

@tool(description="保存用户信息")
def save_user_info(name: str,runtime : ToolRuntime)->str:
    """
    保存用户信息
    """
    runtime.store.put(("users",),runtime.state["user_id"],{"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime)->str:
    """
    获取用户信息

    Returns:
        str:用户信息
    """
    item = runtime.store.get(("users",),runtime.state["user_id"])
    return str(item) if item is not None else "unknow"

class CustomState(AgentState):
    user_id : NotRequired[str]

store = InMemoryStore()

#使用数据库长期记忆
# DB_URL = os.getenv("POSTGRES_DB_URL")
#
# with PostgresStore.from_conn_string(DB_URL) as store:
#     store.setup()

agent = create_agent(
    model = model,
    tools = [save_user_info,get_user_info],
    store = store,
    state_schema = CustomState,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": "你好，很高兴认识你，我是小花",
    "user_id": "user-1"
})

for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": "我是谁",
    "user_id": "user-1"
})

for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你！😊

让我把你的名字记住。
Tool Calls:
  save_user_info (call_00_iLLJPzx0pmvKEwqzAjgR8788)
 Call ID: call_00_iLLJPzx0pmvKEwqzAjgR8788
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

我已经记住你的名字啦，小花！以后我会记住你的。有什么我可以帮你的吗？😊
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

我先查一下您的信息。
Tool Calls:
  get_user_info (call_00_SMN8fWeESj8IbzwkMqC27796)
 Call ID: call_00_SMN8fWeESj8IbzwkMqC2779